# Perplexity / Reward Correlation Analysis

Evaluates model-style sensitivity by computing panel-relative log-probabilities (`s_delta = s_base − median(panel)`) and correlating them with reward-model scores. For each reward model / LM pair it produces a scatter plot and a CSV of Spearman correlations (with cluster-bootstrap 95 % CIs) across all tested language models. The key finding is that reward models are sensitive to model-specific likelihood/style features independent of tokenisation choice or capability level.


In [ ]:
from typing import Dict, Optional, Union, List, Any

from dataclasses import dataclass
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

data_ppl = {
    "gemma-2-2b-it": {"model_id": "google/gemma-2-2b-it", "type": "perplexity", "file": "perplexity_google__gemma-2-2b-it.csv"},
    "gemma-2-9b-it": {"model_id": "google/gemma-2-9b-it", "type": "perplexity", "file": "perplexity_google__gemma-2-9b-it.csv"},
    "gemma-3-12b-it": {"model_id": "google/gemma-3-12b-it", "type": "perplexity", "file": "perplexity_google__gemma-3-12b-it.csv"},
    "Llama-2-13b-chat-hf": {"model_id": "meta-llama/Llama-2-13b-chat-hf", "type": "perplexity", "file": "perplexity_meta-llama__Llama-2-13b-chat-hf.csv"},
    "Llama-2-7b-chat-hf": {"model_id": "meta-llama/Llama-2-7b-chat-hf", "type": "perplexity", "file": "perplexity_meta-llama__Llama-2-7b-chat-hf.csv"},
    "Llama-3.1-8B-Instruct": {"model_id": "meta-llama/Llama-3.1-8B-Instruct", "type": "perplexity", "file": "perplexity_meta-llama__Llama-3.1-8B-Instruct.csv"},
    "Qwen2.5-0.5B-Instruct": {"model_id": "Qwen/Qwen2.5-0.5B-Instruct", "type": "perplexity", "file": "perplexity_Qwen__Qwen2.5-0.5B-Instruct.csv"},
    "Qwen2.5-7B-Instruct": {"model_id": "Qwen/Qwen2.5-7B-Instruct", "type": "perplexity", "file": "perplexity_Qwen__Qwen2.5-7B-Instruct.csv"},
    # "Qwen3-0.6B_thinking": {"model_id": "Qwen/Qwen3-0.6B", "type": "perplexity", "file": "perplexity_Qwen__Qwen3-0.6B_thinking.csv"},
    "Qwen3-0.6B_nothink": {"model_id": "Qwen/Qwen3-0.6B", "type": "perplexity", "file": "perplexity_Qwen__Qwen3-0.6B.csv"},
    # "Qwen3-8B_thinking": {"model_id": "Qwen/Qwen3-8B", "type": "perplexity", "file": "perplexity_Qwen__Qwen3-8B_thinking.csv"},
    "Qwen3-8B_nothink": {"model_id": "Qwen/Qwen3-8B", "type": "perplexity", "file": "perplexity_Qwen__Qwen3-8B.csv"},
}
data_rewards = {
    "RM_Llama-3.1-8B": {"model_id": "allenai/Llama-3.1-8B-Instruct-RM-RB2", "type": "reward", "file": "rewards_allenai__Llama-3.1-8B-Instruct-RM-RB2.csv"},
    "Skywork_Llama-3.1-8B": {"model_id": "Skywork/Skywork-Reward-V2-Llama-3.1-8B", "type": "reward", "file": "rewards_Skywork__Skywork-Reward-V2-Llama-3.1-8B.csv"},
    "Skywork_Qwen3-0.6B": {"model_id": "Skywork/Skywork-Reward-V2-Qwen3-0.6B", "type": "reward", "file": "rewards_Skywork__Skywork-Reward-V2-Qwen3-0.6B.csv"},
    "Skywork_Qwen3-8B": {"model_id": "Skywork/Skywork-Reward-V2-Qwen3-8B", "type": "reward", "file": "rewards_Skywork__Skywork-Reward-V2-Qwen3-8B.csv"},
    "RM_Deberta": {"model_id": "OpenAssistant/reward-model-deberta-v3-large-v2", "type": "reward", "file": "rewards_OpenAssistant__reward-model-deberta-v3-large-v2.csv"},
}

path_to_data = "perplexity_data/"

@dataclass
import yaml
from pathlib import Path


In [ ]:
class AnalysisPair:
    base_key: str
    models: List[str]
    reward: Dict[str, Any]
    family_name: Optional[str] = None

analysis_pairs = [
    AnalysisPair(
        base_key="Llama-3.1-8B-Instruct",
        models=[k for k in data_ppl], # [k for k in data_ppl if k != "RM_Llama-3.1-8B"],
        reward=data_rewards["RM_Llama-3.1-8B"],
        family_name="Llama",
    ),
    AnalysisPair(
        base_key="Llama-3.1-8B-Instruct",
        models=[k for k in data_ppl],
        reward=data_rewards["Skywork_Llama-3.1-8B"],
        family_name="Llama",
    ),
    AnalysisPair(
        base_key="Qwen3-8B_thinking", # "Qwen3-8B_nothink",
        models=[k for k in data_ppl],
        reward=data_rewards["Skywork_Qwen3-8B"],
        family_name="Qwen",
    ),
    AnalysisPair(
        base_key="Qwen3-0.6B_thinking", #, "Qwen3-0.6B_nothink",
        models=[k for k in data_ppl],
        reward=data_rewards["Skywork_Qwen3-0.6B"],
        family_name="Qwen",
    ),
    AnalysisPair(
        base_key="None",
        models=[k for k in data_ppl],
        reward=data_rewards["RM_Deberta"],
    ),
]


In [ ]:
def bootstrap_spearman_ci(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    *,
    group_col: str = "dataset_id",
    n_boot: int = 2000,
    alpha: float = 0.05,
    seed: int = 0,
) -> Dict[str, float]:
    """
    Cluster bootstrap Spearman correlation with percentile CI.
    Resamples group_col (dataset_id) with replacement.
    """
    rng = np.random.default_rng(seed)
    groups = df[group_col].unique()
    if len(groups) < 2:
        return {"rho": np.nan, "ci_lo": np.nan, "ci_hi": np.nan}

    # Point estimate
    rho = float(df[x_col].corr(df[y_col], method="spearman"))

    boot = np.empty(n_boot, dtype=np.float64)
    for b in range(n_boot):
        sampled = rng.choice(groups, size=len(groups), replace=True)
        # keep both chosen/rejected rows for each sampled id
        bs = df[df[group_col].isin(sampled)]
        boot[b] = bs[x_col].corr(bs[y_col], method="spearman")

    lo = float(np.nanquantile(boot, alpha / 2))
    hi = float(np.nanquantile(boot, 1 - alpha / 2))
    return {"rho": rho, "ci_lo": lo, "ci_hi": hi}


In [ ]:
def _load_defaults(section: str) -> dict:
    """Return default config values for *section* from configs/default_values.yaml."""
    defaults_path = Path(__file__).resolve().parent / "configs" / "default_values.yaml"
    with open(defaults_path) as f:
        return yaml.safe_load(f).get(section, {})


def run(config_path: "str | Path | None" = None, **overrides):
    """Run perplexity / reward correlation analysis.

    Args:
        config_path: Optional YAML config to override defaults.  If relative,
                     resolved from the ``configs/`` directory next to this notebook.
        **overrides: Keyword overrides, e.g. ``corr_kind="pearson"``.
    """
    cfg = _load_defaults("eval_perplexity")

    if config_path is not None:
        p = Path(config_path)
        if not p.is_absolute():
            p = Path(__file__).resolve().parent / "configs" / p
        with open(p) as f:
            cfg.update(yaml.safe_load(f))

    cfg.update(overrides)

    NLL_NORM = cfg["nll_norm"]
    CORR_KIND = cfg["corr_kind"]
    DO_FAMILY_SPLIT = cfg["do_family_split"]
    PANEL_FUNC = cfg["panel_func"]


NLL_NORM = "bytes" # use  "token", "bytes", "chars"
CORR_KIND = "spearman"   # use "pearson" or "spearman"
DO_FAMILY_SPLIT = False
PANEL_FUNC = "median" # use "mean or "median"


print("-------------------------------")
# Do analysis here.
for a_pair in analysis_pairs:
    # ---- Load reward + perplexity data once per analysis pair ----
    reward_df = pd.read_csv(path_to_data + a_pair.reward["file"])
    reward_df = reward_df[["dataset_id", "which", "reward"]].dropna()
    reward_s = reward_df.set_index(["dataset_id", "which"])["reward"]

    # Sanity: reward keys should be unique
    if not reward_s.index.is_unique:
        dup = reward_s.index[reward_s.index.duplicated()].unique()
        raise ValueError(f"Reward file has duplicate (dataset_id, which) keys, e.g. {list(dup[:5])}")

    # Load all perplexity CSVs into one wide table: index=(dataset_id, which), columns=model_key, values=s
    ppl_long_parts = []
    for k, cfg in data_ppl.items():
        df = pd.read_csv(path_to_data + cfg["file"])

        needed = {"dataset_id", "which", "nll_sum", "nll_mean_token", "norm_n_y_bytes", "norm_n_y_chars"}
        missing = needed - set(df.columns)
        if missing:
            raise KeyError(f"{cfg['file']} missing required columns: {sorted(missing)}")

        df = df[["dataset_id", "which", "nll_sum", "nll_mean_token", "norm_n_y_bytes", "norm_n_y_chars"]].copy()

        if NLL_NORM == "token":
            df["s"] = -df["nll_mean_token"]
        elif NLL_NORM == "bytes":
            denom = df["norm_n_y_bytes"].astype(float)
            df["s"] = -(df["nll_sum"].astype(float) / denom.replace(0.0, np.nan))
        elif NLL_NORM == "chars":
            denom = df["norm_n_y_chars"].astype(float)
            df["s"] = -(df["nll_sum"].astype(float) / denom.replace(0.0, np.nan))
        else:
            raise ValueError(f"Unknown NLL_NORM={NLL_NORM!r}. Use 'token', 'bytes', or 'chars'.")

        df["model_key"] = k
        ppl_long_parts.append(df[["dataset_id", "which", "model_key", "s"]])

    ppl_long = pd.concat(ppl_long_parts, ignore_index=True)
    ppl_wide = ppl_long.pivot_table(
        index=["dataset_id", "which"],
        columns="model_key",
        values="s",
        aggfunc="first",
    )

    # Sanity: perplexity keys should be unique (pivot_table with 'first' can hide duplicates)
    if ppl_long.duplicated(subset=["dataset_id", "which", "model_key"]).any():
        raise ValueError("Perplexity data has duplicate (dataset_id, which, model_key) rows; pivot_table would hide this.")


    # ---- Build family mapping from data_ppl metadata (best-effort) ----
    def _infer_family(model_id: str) -> str:
        mid = model_id.lower()
        if "llama" in mid:
            return "Llama"
        if "qwen" in mid:
            return "Qwen"
        if "gemma" in mid:
            return "Gemma"
        return "Other"

    model_key_to_family = {k: _infer_family(cfg["model_id"]) for k, cfg in data_ppl.items()}
    family = a_pair.family_name  # e.g., "Llama" or "Qwen"

    # Prepare two plots per reward model: (A) full panel, (B) other-family-only panel
    plot_specs = [
            ("all_models_panel", "Panel = all other models", None),  # None => use all except m
        ]
    if DO_FAMILY_SPLIT:
        plot_specs.append(("other_family_panel", "Panel = only other-family models", "other_family_only"))

    for plot_tag, plot_title_suffix, panel_mode in plot_specs:
        fig, ax = plt.subplots(figsize=(10, 6))
        corr_rows = []

        for m in a_pair.models:
            # ---- Panel construction ----
            panel_all = [k for k in data_ppl if k != m]

            if panel_mode is None:
                panel = panel_all
            else:
                # Only models NOT in the same family as a_pair.family_name
                # (and still exclude m, even if it is not in-family)
                panel = [k for k in panel_all if model_key_to_family.get(k, "Other") != family]

            # Need base model column + at least 1 panel column
            if m not in ppl_wide.columns:
                continue
            panel = [p for p in panel if p in ppl_wide.columns]
            if len(panel) == 0:
                continue

            # Compute s_delta on intersection of available rows
            if PANEL_FUNC == "mean":
                s_delta = ppl_wide[m] - ppl_wide[panel].mean(axis=1)
            elif PANEL_FUNC == "median":
                s_delta = ppl_wide[m] - ppl_wide[panel].median(axis=1)
            else:
                raise ValueError(f"Unknown PANEL_FUNC={PANEL_FUNC!r}. Use 'mean' or 'median'.")

            # Join with reward scores; drop NaNs/infs
            tmp = pd.concat([s_delta.rename("s_delta"), reward_s.rename("reward")], axis=1, join="inner")
            tmp = tmp.replace([np.inf, -np.inf], np.nan).dropna(subset=["s_delta", "reward"])
            if len(tmp) == 0:
                continue

            # tmp currently has index=(dataset_id, which). Turn dataset_id into a column for bootstrapping.
            tmp2 = tmp.reset_index()  # gives columns: dataset_id, which, s_delta, reward

            if CORR_KIND == "spearman":
                ci = bootstrap_spearman_ci(
                    tmp2, "s_delta", "reward",
                    group_col="dataset_id",
                    n_boot=2000,
                    alpha=0.05,
                    seed=0,
                )
                corr = ci["rho"]
                corr_rows.append({
                    "model": m,
                    "spearman_r": corr,
                    "ci95_lo": ci["ci_lo"],
                    "ci95_hi": ci["ci_hi"],
                    "n": int(len(tmp2)),
                    "n_ids": int(tmp2["dataset_id"].nunique()),
                })
            else:
                corr = float(tmp["s_delta"].corr(tmp["reward"], method=CORR_KIND))
                corr_rows.append({"model": m, f"{CORR_KIND}_r": corr, "n": int(len(tmp))})


            ax.scatter(
                tmp["s_delta"].to_numpy(),
                tmp["reward"].to_numpy(),
                s=10,
                alpha=0.35,
                label=f"{m} ({CORR_KIND[0]}={corr:.3f}, n={len(tmp)})",
            )

        ax.set_xlabel(f"Panel-relative log-prob (s_delta = s_base - {PANEL_FUNC}(panel)) [ ]")
        ax.set_ylabel("Reward model score [ ]")
        ax.set_title(
            f"Reward vs panel-relative perplexity: {a_pair.base_key} | {plot_title_suffix} | "
            f"NLL_NORM={NLL_NORM} | CORR={CORR_KIND}"
        )
        ax.set_xlim(-3., 3.)
        ax.grid(True, alpha=0.2)
        if len(corr_rows) > 0:
            ax.legend(fontsize=8, loc="best", frameon=True)
        plt.tight_layout()

        out_plot = (
            f"scatter_reward_vs_sdelta__{a_pair.reward['model_id'].replace('/','__')}__"
            f"{a_pair.base_key}__{plot_tag}__{NLL_NORM}__{CORR_KIND}.png"
        )
        plt.savefig(out_plot, dpi=300)

        corr_df = pd.DataFrame(corr_rows).sort_values(f"{CORR_KIND}_r", ascending=False)
        corr_df["ci95_minus"] = corr_df["spearman_r"] - corr_df["ci95_lo"]
        corr_df["ci95_plus"] = corr_df["ci95_hi"] - corr_df["spearman_r"]
        out_corr = (
            f"corr_reward_vs_sdelta__{a_pair.reward['model_id'].replace('/','__')}__"
            f"{a_pair.base_key}__{plot_tag}__{NLL_NORM}__{CORR_KIND}.csv"
        )
        corr_df.to_csv(out_corr, index=False)

        print(f"\nReward Model {a_pair.reward['model_id']}")
        mean_abs_corr = corr_df["spearman_r"].abs().mean()
        print(f"Mean absolute correlation {mean_abs_corr}")
        print(corr_df)


In [ ]:
run()
